<div style="text-align: center;">

    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>

</div>

<div style="text-align: center;">

    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">

</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>



`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.



- **What Does PyPDFLoader Do?**

  - Extracts text from PDF files, retaining formatting and layout.

  - Simplifies the preprocessing of document-based datasets.

  - Supports efficient and scalable loading of large PDF collections.



- **Key Features:**

  - Compatible with popular NLP libraries and frameworks.

  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).

  - Provides flexible configurations for structured text extraction.



- **Use Cases:**

  - Preparing PDF documents for retrieval-based systems in RAGs.

  - Automating the text extraction pipeline for document analysis.

  - Creating datasets from academic papers, technical manuals, and reports.


In [1]:
# If running in Colab, upload the PDF book to this session first
from google.colab import files
uploaded = files.upload()  # select ai-for-everyone.pdf

Saving ai-for-everyone.pdf to ai-for-everyone (2).pdf


In [2]:
%pip install langchain langchain_community pypdf

%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')

/tmp/ipykernel_5652/3130841623.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [4]:
# File path for the document

file_path = "ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>



The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.



This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [5]:
# Load and split the document

loader = PyPDFLoader(file_path)

pages = loader.load_and_split()

len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>





####  RecursiveCharacterTextSplitter in LangChain



The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.



####  Parameters



| Parameter       | Description                                                                 |

|-----------------|-----------------------------------------------------------------------------|

| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |

| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |



####  How it works

`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:

1. Paragraphs (`"\n\n"`)

2. Lines (`"\n"`)

3. Sentences or words (`" "`)

4. Individual characters (as a last resort)



This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.




In [6]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200

)

chunks = text_splitter.split_documents(pages)



len(chunks)

1096

####  Alternative: CharacterTextSplitter



`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.



##### Example:

```python

from langchain.text_splitter import CharacterTextSplitter



text_splitter = CharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200

)

````



This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.



---



#### Comparison Table



| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |

| ------------------------------ | ------------------------------ | ------------------------- |

| Structure-aware splitting      |  Yes                          |  No                      |

| Preserves sentence/paragraphs  |  Yes                          |  No                      |

| Risk of splitting mid-sentence |  Minimal                     |  High                   |

| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |

| Performance speed              |  Slightly slower             |  Faster                  |



---



#### Recommendation



Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG



### Best Practices for Chunk Size in RAG



| Factor                      | Recommendation                                                                                                                                                                                          |

| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |

| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |

| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |

| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |

| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |

| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |

| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |

| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |

| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |





### Rule of Thumb



| Use Case                 | Chunk Size      | Overlap |

| ------------------------| --------------- | ------- |

| Factual Q&A              | 500–800 chars   | 100–200 |

| Summarization            | 1000–1500 chars | 200–300 |

| Technical documents      | 400–700 chars   | 100–200 |

| Long reports/books       | 800–1200 chars  | 200–300 |

| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |





### Avoid



- Chunks >2000 characters: risks context overflow.

- No overlap: may lose key information between chunks.




<h2 style="color: #FF8C00;">Embeddings</h2>



Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.



- **What are OpenAI Embeddings?**

  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.

  - Encapsulate semantic relationships in the text, enabling robust NLP applications.



- **Key Features of `text-embedding-3-large`:**

  - Large-scale embedding model optimized for accuracy and versatility.

  - Handles diverse NLP tasks, including retrieval, classification, and clustering.

  - Ideal for applications with high-performance requirements.



- **Benefits:**

  - Reduces the need for extensive custom training.

  - Provides state-of-the-art performance in retrieval-augmented systems.

  - Compatible with RAGs to create powerful context-aware models.


In [7]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [8]:
load_dotenv()

False

In [9]:
# If running in Colab, you likely don't have a local .env file.
# Option A: use Colab's Secrets manager (recommended) - click the key icon on the left sidebar,
# add a secret named OPENAI_API_KEY, then uncomment the two lines below:
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Option B: set it directly for this session only (do NOT commit this to git with your real key!)
# os.environ["OPENAI_API_KEY"] = "sk-..."

In [15]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [16]:
api_key = os.getenv("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>



ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.



### Workflow Overview:

- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).

- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.

- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.



### Key Features of ChromaDB:

- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.

- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.

- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [17]:
from langchain_chroma import Chroma

In [18]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")

print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [19]:
user_question = "What are the main criticisms of AI development raised in this book, particularly regarding power and inequality?"
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [20]:
# Display top results

for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results

    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
 accessible for everyone. Further in this intro-
duction, I elaborate on this and I make the case for a radical democratisation of 
AI, and why we need to put power at the centre for achieving this.
The contributions in this book braid discussion of power and critique with 
three strands: AI – Humans vs. Machines, Discourses and Myths About AI and 
AI Power and Inequalities. 
Part 1: AI – Humans vs. Machines – deals with the history and conceptualisa-
tion of AI and what is at stake in its development. This section looks at different 
perspectives about what characterises machine intelligence and how it might 
be important to further radical humanism in the era of automation and AI. 
Part 2: Discourses and Myths About AI – analyses how AI is framed in popu-
lar and scholarly discussions and investigates the normative projections of what 
AI should be and what it should do. This section poses critical questions about
Document 2:
vestigates the normative projections of what 


<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [21]:
def _get_document_prompt(docs):

    prompt = "\n"

    for doc in docs:

        prompt += "\nContent:\n"

        prompt += doc.page_content + "\n\n"

    return prompt

In [22]:
# Generate a formatted context from the retrieved documents

formatted_context = _get_document_prompt(retrieved_docs)

print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [23]:
prompt = f"""
You are a knowledgeable assistant helping a reader understand the book "AI for Everyone? Critical Perspectives",
an academic text examining the social, political, and ethical implications of Artificial Intelligence.

Using ONLY the context provided below (retrieved from the book), answer the user's question in a clear,
well-structured way. Adopt an academic but accessible tone, consistent with the book's critical perspective
on AI, power, and inequality. If the context does not contain enough information to answer the question,
say so honestly instead of making up an answer.

Context:
{formatted_context}

Question:
{user_question}

Answer:
"""

In [24]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [25]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.3,       # Lower temperature: keep answers grounded in the retrieved context, not overly creative
    'max_tokens': 600,        # Allow for a full, well-developed academic-style answer
    'top_p': 0.9,             # Nucleus sampling: slight diversity while staying focused
    'frequency_penalty': 0.3, # Reduce repetition of the same phrases
    'presence_penalty': 0.2   # Mild encouragement to cover different points from the context, not just repeat one
}

<h1 style="color: #FF6347;">Response</h1>


In [26]:
messages = [{'role': 'user', 'content': prompt}]

completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [27]:
answer = completion.choices[0].message.content

print(answer)

The book "AI for Everyone? Critical Perspectives" raises several key criticisms of AI development, particularly concerning issues of power and inequality. These criticisms are woven throughout the text and can be summarized as follows:

1. **Concentration of Power**: The book argues that the benefits of AI are not equally accessible to everyone, largely due to the concentration of power among a few dominant tech companies. These companies possess the computational power, financial resources, and access to large datasets necessary for developing advanced AI systems. This concentration raises critical questions about who ultimately benefits from AI technologies.

2. **Inequality and Exclusion**: There is a significant concern about new inequalities that AI might bring about. The book critically examines how AI capitalism is organized and highlights the risks of exclusion, questioning whether AI truly serves everyone or primarily those with decision-making power.

3. **Lack of Diverse Per

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>



**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:



- **-1**: Vectors are completely opposite.

- **0**: Vectors are orthogonal (uncorrelated or unrelated).

- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>



Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [28]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [29]:
def highlight_keywords(text, keywords):

    for keyword in keywords:

        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))

    return text

### Exercice4: add your keywords

In [30]:
query_keywords = ["power", "inequality", "AI", "critical", "bias"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
ing sure that the benefits of AI are accessible for everyone. Further in this intro-
duction, I elaborate on this and I make the case for a radical democratisation of 
AI, and why we need to put power
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.

2. The loop iterates over the first document in retrieved_docs.

3. For each document, a snippet of the first 200 characters is extracted.

4. The highlight_keywords function is called to highlight the keywords in the snippet.

5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:
